# VIVA AI — Plant Disease Classifier Training v2 (Colab, free GPU tier)

**Rebuilt with the same two fixes as the soil notebook:** robust
auto-discovery of the dataset's real folder structure instead of
hardcoded assumptions, and training/conversion split across a runtime
restart to avoid the `tensorflowjs` + TensorFlow environment conflict
that's a common, well-documented cause of Colab notebooks silently
breaking partway through.

**Dataset:** `vipoooool/new-plant-diseases-dataset` on Kaggle — the
standard PlantVillage republication, actively cited in recent published
research, ~87k images across 38 classes. This notebook filters it down
to a small, visually distinct 4-class subset by default — per the
project spec, a narrow accurate model beats a broad unreliable one for a
live demo.

**Before running:** Runtime → Change runtime type → T4 GPU (free tier).


## 1. Get a free Kaggle API token

Same as the soil notebook: kaggle.com → Account → "Create New API Token" → upload the downloaded `kaggle.json` below.

In [ ]:
from google.colab import files
print("Upload your kaggle.json (from Kaggle > Account > Create New API Token)")
uploaded = files.upload()


In [ ]:
import os
os.makedirs("/root/.kaggle", exist_ok=True)
os.system("cp kaggle.json /root/.kaggle/kaggle.json")
os.system("chmod 600 /root/.kaggle/kaggle.json")
print("Kaggle credentials installed.")


## 2. Download the dataset

~2.7GB, takes a few minutes — that's normal, don't cancel it.

In [ ]:
!pip install -q kaggle
import subprocess

result = subprocess.run(
    ["kaggle", "datasets", "download", "-d", "vipoooool/new-plant-diseases-dataset",
     "-p", "/content/disease_raw", "--unzip"],
    capture_output=True, text=True,
)
if result.returncode != 0:
    raise RuntimeError(
        f"Download failed: {result.stderr.strip()[-500:]}\n\n"
        "If this dataset has also been removed/renamed by the time you're "
        "reading this, search kaggle.com for 'plant disease classification', "
        "pick a dataset with Crop___Disease-style folder names (or similar "
        "labeled structure), and swap the slug above."
    )
print("Download succeeded.")


## 3. Auto-discover the real folder structure

Same principle as the soil notebook: don't assume a fixed path like
`/content/disease_raw/.../train` exists — search for it, and print what's
actually there so a structure change doesn't fail silently or
cryptically.

In [ ]:
import os, glob

IMAGE_EXTENSIONS = (".jpg", ".jpeg", ".png", ".bmp", ".webp")

def find_class_folders(root):
    """PlantVillage-style datasets name leaf folders 'Crop___Disease'.
    Find every folder matching that pattern that contains image files,
    at any depth."""
    found = {}
    for dirpath, _, filenames in os.walk(root):
        name = os.path.basename(dirpath)
        images = [f for f in filenames if f.lower().endswith(IMAGE_EXTENSIONS)]
        if images and ("___" in name or len(images) > 50):
            found[dirpath] = len(images)
    return found

class_folders = find_class_folders("/content/disease_raw")
print(f"Found {len(class_folders)} class-like folders. Showing top 15 by image count:\n")
for path, count in sorted(class_folders.items(), key=lambda x: -x[1])[:15]:
    print(f"  {count:5d} images  —  {os.path.basename(path)}")

assert class_folders, (
    "No class folders found. Run '!find /content/disease_raw -maxdepth 4 -type d' "
    "in a new cell to inspect the actual structure and adjust find_class_folders() above."
)


## 4. Select your 4 demo classes

Default targets a tomato/pepper subset for the standard PlantVillage
folder names. **If step 3's printed list shows different folder names**
(dataset structure changed), edit `SELECTED_CLASSES` below to match
what's actually there — copy exact folder names from the printed list.

In [ ]:
SELECTED_CLASSES = {
    "healthy":      ["Tomato___healthy"],
    "leaf_blight":  ["Tomato___Early_blight"],
    "yellowing":    ["Tomato___Tomato_Yellow_Leaf_Curl_Virus"],
    "pest_damage":  ["Pepper,_bell___Bacterial_spot"],
}
CLASS_ORDER = list(SELECTED_CLASSES.keys())  # fixed order, matches DISEASE_LABELS in mockModel.ts

# Verify every targeted folder name actually exists among what was discovered —
# fail clearly here rather than training on silently-empty classes.
available_names = {os.path.basename(p) for p in class_folders}
for canonical, names in SELECTED_CLASSES.items():
    if not any(n in available_names for n in names):
        print(f"WARNING: none of {names} found for '{canonical}'. "
              f"Pick a real folder name from the step 3 list and update SELECTED_CLASSES.")


## 5. Copy into a clean dataset folder

In [ ]:
import shutil

DST_ROOT = "/content/dataset"
os.makedirs(DST_ROOT, exist_ok=True)

path_by_name = {os.path.basename(p): p for p in class_folders}

for canonical, candidate_names in SELECTED_CLASSES.items():
    dst_dir = os.path.join(DST_ROOT, canonical)
    os.makedirs(dst_dir, exist_ok=True)
    count = 0
    for name in candidate_names:
        src_dir = path_by_name.get(name)
        if not src_dir:
            continue
        for f in os.listdir(src_dir)[:600]:  # cap per source to keep training fast
            if f.lower().endswith(IMAGE_EXTENSIONS):
                shutil.copy(os.path.join(src_dir, f), dst_dir)
                count += 1
    print(f"{canonical}: {count} images copied")

empty_classes = [c for c in CLASS_ORDER if not os.listdir(os.path.join(DST_ROOT, c))]
assert not empty_classes, (
    f"These classes ended up with zero images: {empty_classes}. Fix "
    f"SELECTED_CLASSES in the previous cell using real folder names from "
    f"step 3's printed list, then re-run from this cell."
)


## 6. Train MobileNetV2 transfer-learning model

Same as the soil notebook: deliberately doesn't touch the pre-installed TensorFlow version.

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.preprocessing.image import ImageDataGenerator

print("TensorFlow version (Colab's pre-installed build, untouched):", tf.__version__)

IMG_SIZE = (224, 224)
BATCH_SIZE = 16
EPOCHS = 12

train_gen = ImageDataGenerator(
    rescale=1.0 / 255, rotation_range=15, width_shift_range=0.1,
    height_shift_range=0.1, horizontal_flip=True,
    brightness_range=[0.8, 1.2], validation_split=0.2,
)

train_data = train_gen.flow_from_directory(
    DST_ROOT, target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode="categorical", subset="training", classes=CLASS_ORDER,
)
val_data = train_gen.flow_from_directory(
    DST_ROOT, target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode="categorical", subset="validation", classes=CLASS_ORDER,
)
print("Class order (fixed, matches DISEASE_LABELS order in mockModel.ts):", CLASS_ORDER)


In [ ]:
base_model = MobileNetV2(input_shape=(*IMG_SIZE, 3), include_top=False, weights="imagenet")
base_model.trainable = False

model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(64, activation="relu"),
    layers.Dropout(0.3),
    layers.Dense(4, activation="softmax"),
])

model.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"])
history = model.fit(train_data, validation_data=val_data, epochs=EPOCHS)


## 7. Check validation accuracy, fine-tune if needed

In [ ]:
val_loss, val_acc = model.evaluate(val_data)
print(f"Validation accuracy: {val_acc:.2%}")

if val_acc < 0.75:
    print("Fine-tuning: unfreezing the last 20 layers of MobileNetV2...")
    base_model.trainable = True
    for layer in base_model.layers[:-20]:
        layer.trainable = False
    model.compile(optimizer=tf.keras.optimizers.Adam(1e-5),
                   loss="categorical_crossentropy", metrics=["accuracy"])
    model.fit(train_data, validation_data=val_data, epochs=5)
    val_loss, val_acc = model.evaluate(val_data)
    print(f"Validation accuracy after fine-tuning: {val_acc:.2%}")


## 8. Save the model to disk — STOP after this cell

In [ ]:
model.save("/content/disease_model_keras.h5")
print("Saved to /content/disease_model_keras.h5")

import json as _json
with open("/content/disease_class_order.json", "w") as f:
    _json.dump(CLASS_ORDER, f)
print("Class order saved for after the restart:", CLASS_ORDER)


## 9. ⚠️ Restart the runtime now, then continue below

**Runtime menu → Restart session** (NOT "Disconnect and delete
runtime" — that wipes the disk and your saved model with it). This
clears Python's memory but keeps `/content/` files intact, and isolates
the upcoming `tensorflowjs` install from the TensorFlow build used for
training — the exact thing that broke the previous version of this
notebook.

In [ ]:
# Run this AFTER Runtime > Restart session.
import json as _json, os

with open("/content/disease_class_order.json") as f:
    CLASS_ORDER = _json.load(f)
print("Reloaded class order:", CLASS_ORDER)

assert os.path.exists("/content/disease_model_keras.h5"), (
    "disease_model_keras.h5 not found — if you fully disconnected the "
    "runtime instead of using 'Restart session', the disk was wiped and "
    "training needs to be re-run from the top."
)
print("Found saved model on disk, proceeding to convert.")


## 10. Install tensorflowjs fresh, in this clean session, and convert

In [ ]:
!pip install -q tensorflowjs

!tensorflowjs_converter --input_format=keras \
    /content/disease_model_keras.h5 \
    /content/disease_model_tfjs

print("Conversion complete. Files:")
!ls -la /content/disease_model_tfjs


## 11. Download and install into the app

Copy every file from the downloaded zip into
`public/models/disease_model/`, replacing the placeholder README there.
`CLASS_ORDER` (`healthy, leaf_blight, yellowing, pest_damage`) already
matches `DISEASE_LABELS` in `src/lib/mockModel.ts` exactly — no
relabeling step needed, since it was fixed explicitly in code rather
than left to alphabetical folder ordering.

In [ ]:
import shutil
from google.colab import files
shutil.make_archive("/content/disease_model_tfjs", "zip", "/content/disease_model_tfjs")
files.download("/content/disease_model_tfjs.zip")
